In [3]:
import duckdb
import pandas as pd
source = r"C:\Users\eddiec11us\Downloads\Upload Source File.xlsx"

df = pd.read_excel(source)
conn = duckdb.connect()
conn.register("source", df)

In [44]:
sql = """
WITH included_price_parts AS (
SELECT DISTINCT
  "Inventory ID" AS part_number
FROM source
WHERE "Price Type" = 'Customer'

UNION

SELECT DISTINCT
  "Inventory ID" AS part_number
FROM source
WHERE "Price Type" = 'Customer Price Class'
AND "Price Code" IN ('DEAL','PART','DIST','SPEC')
),

msrp_rows AS (
SELECT DISTINCT
  "Inventory ID" AS part_number
FROM source
WHERE "Price Type" = 'Base'
)

SELECT
  part_number
FROM msrp_rows base
WHERE EXISTS (
  SELECT 1
  FROM included_price_parts i
  WHERE i.part_number = base.part_number
  )
"""
# show all base parts that either have a contract price or a price level



In [45]:
msrp_to_upload = conn.sql(sql).df()
msrp_to_upload.to_csv(r"C:\Users\eddiec11us\Downloads\MSRP Parts To Upload.csv")